# RoBERTA on agnews
### LLM data generation: **"deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"**

https://huggingface.co/FacebookAI/roberta-base  
https://huggingface.co/FacebookAI/roberta-large

- total number of train samples: 500
- total number of test&nbsp; samples: 500
- classification model: "roberta-large"
- dataset: agnews

Generation methods:
- **A** generic augmentation
- **B** targeted augmentation
- **C** unsupervised context augmentation
- **D** real data with labels generated in zero-shot settings

TESTS:
1. 500 real
2. 250 real + 250 synthetic (A, B, C, D) [50% synthetic]
3. 50 real + 450 synthetic (A, B, C, D) [90% synthetic]
4. 500 synthetic (A, B, C, D) [100% synthetic]

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import json
from sklearn.model_selection import train_test_split

# CHANGE WORKING DIRECTORY TO ROOT
current_dir = os.path.basename(os.getcwd())
if current_dir == "src":
    os.chdir("..") # Move up by 1
elif os.path.basename(os.getcwd()) == "bai-thesis-nlp":  
    pass # If already at root, stay there
else:
    os.chdir("../..") # Move up by 2 otherwise
     
from src._utils._helpers import get_generated_examples_df
from src._utils._data_analysis_helpers import plot_micromacrof1
from src._utils._run_multiclassRoBERTA import main_multiclassRoBERTA

In [ ]:
LLM_NAME = "DeepSeek-R1-Distill-Qwen-1.5B"
DATASET_NAME = "agnews"
FOLDER_DIR = "src/"+DATASET_NAME+"/experiments/RoBERTA_500samples/data_"+LLM_NAME
os.makedirs(FOLDER_DIR, exist_ok=True)
LOG_DIR = os.path.join(FOLDER_DIR, "RoBERTA_log.json")

### LOAD DATA ###
# real data
real_train_df = pd.read_csv("real_data/train/"+DATASET_NAME+"trainAll.csv").rename(columns={"2": "text", "3": "label"})
real_train_df.drop(columns=["0", "1"], inplace=True)

# Take 500 samples for dev set
real_train_df, dev_df = train_test_split(real_train_df, test_size=500, random_state=42, stratify=real_train_df["label"])
# Store the dev set into a CSV file
dev_df.to_csv(os.path.join(FOLDER_DIR, "dev_set.csv"), index=False)

# synthetic data (generic, targeted and unsupervised context)
SYNTHETIC_DATA_DIR = "synthetic_data/datasets/"+LLM_NAME+"/"
syn_generic_df, _ = get_generated_examples_df(SYNTHETIC_DATA_DIR+DATASET_NAME+"_baseline_500.json")
syn_targeted_df, _ = get_generated_examples_df(SYNTHETIC_DATA_DIR+DATASET_NAME+"_targeted+tags_500.json")
syn_targeted_df = syn_targeted_df.drop(columns=["phenomena"])
syn_unsupContext_df, _ = get_generated_examples_df(SYNTHETIC_DATA_DIR+DATASET_NAME+"_unsupervisedContext_500.json")
syn_unsupContext_df = syn_unsupContext_df.drop(columns=["context_examples"])

In [ ]:
# zeroshot data (real data + zeroshot generated labels)
data = "./src/"+DATASET_NAME+"/experiments/fewshotCasualLM_500samples/"+LLM_NAME+"_zeroshot_TRAIN.csv"
zeroshot_df = pd.read_csv(data)

display(zeroshot_df.head())
acc = sum(zeroshot_df['label'] == zeroshot_df['predicted_label']) / len(zeroshot_df)
print(f"Accuracy {acc:.3f} | number of samples: {zeroshot_df.shape[0]}"),

# keep only the labels that are allowed
labels = zeroshot_df['label'].unique()
zeroshot_df = zeroshot_df[zeroshot_df["predicted_label"].isin(labels)].reset_index(drop=True)
new_acc = sum(zeroshot_df['label'] == zeroshot_df['predicted_label']) / len(zeroshot_df)

# set the predicted label as the real label for training
zeroshot_df['label'] = zeroshot_df["predicted_label"]
zeroshot_df = zeroshot_df.drop(columns=["predicted_label"])

print("Filter out labels that are not in the real data. Results:")
print(f"Accuracy {new_acc:.3f} | number of samples: {zeroshot_df.shape[0]}")

In [ ]:
results = []

def get_results(details):
    """Just get useful results from the details"""
    res = {}
    res["generation_method"] = details["generation_method"]
    res["synthetic_ratio"] = details["synthetic_ratio"]
    for key, value in details["metrics_dev"].items():
        res[key] = value
    res["train_time"] = details["train_time"]
    res["eval_time"] = details["eval_time"]
    
    df = pd.DataFrame([res])
    display(df.round(3))
    return df

In [ ]:
base_config = {
    "real_df": real_train_df,
    # "synth_df": None/syn_generic_df..,
    "dev_df": dev_df,
    # "synth_ratio": 0.0/0.5..,
    "max_samples": 500,
    "epochs": 8,
    "batch_size": 16,
    "output_dir":FOLDER_DIR,
    "log_dir": LOG_DIR,
    # "generation_method": None/"generic"/"targeted",
    "save_model": False,
    "save_dataset": True,
}

## 1. 500 real

In [ ]:
config = base_config.copy()
config["synth_df"] = None
config["synth_ratio"] = 0.0
config["generation_method"] = None

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

## 2. 250 real + 250 synthetic

A. Generic augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_generic_df # Generic Augmentation
config["synth_ratio"] = 0.5 # 50% of the data is synthetic
config["generation_method"] = "generic"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

B. Targeted augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_targeted_df, # Targeted Augmentation
config["synth_ratio"] = 0.5 # 50% of the data is synthetic
config["generation_method"] = "targeted"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

C. Unsupervised Context augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_unsupContext_df # Unsupervised Context Augmentation
config["synth_ratio"] = 0.5 # 50% of the data is synthetic
config["generation_method"] = "unsupContext"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

D. Zeroshot labels

In [ ]:
config = base_config.copy()
config["synth_df"] = zeroshot_df # real data with labels predicted in zero-shot setting
config["synth_ratio"] = 0.5 # 50% of the data is synthetic
config["generation_method"] = "zeroshotLabels"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

## 3. 50 real + 450 synthetic
A. Generic augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_generic_df # Generic Augmentation
config["synth_ratio"] = 0.9 # 90% of the data is synthetic
config["generation_method"] = "generic"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

B. Targeted augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_targeted_df # Targeted Augmentation
config["synth_ratio"] = 0.9 # 90% of the data is synthetic
config["generation_method"] = "targeted"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

C. Unsupervised Context augmentation

In [ ]:
config = base_config.copy()
config["synth_df"] = syn_unsupContext_df # Unsupervised Context Augmentation
config["synth_ratio"] = 0.9 # 90% of the data is synthetic
config["generation_method"] = "unsupContext"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

D. Zeroshot labels

In [ ]:
config = base_config.copy()
config["synth_df"] = zeroshot_df # real data with labels predicted in zero-shot setting
config["synth_ratio"] = 0.9 # 90% of the data is synthetic
config["generation_method"] = "zeroshotLabels"

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

## 4. 500 synthetic
A. Generic augmentation

In [ ]:
config = base_config.copy()
config["real_df"] = None
config["synth_df"] = syn_generic_df # Generic Augmentation
config["synth_ratio"] = 1.0 # 100% of the data is synthetic
config["generation_method"] = "generic"
config["save_dataset"] = False

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

B. Targeted augmentation

In [ ]:
config = base_config.copy()
config["real_df"] = None
config["synth_df"] = syn_targeted_df # Targeted Augmentation
config["synth_ratio"] = 1.0 # 100% of the data is synthetic
config["generation_method"] = "targeted"
config["save_dataset"] = False

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

C. Unsupervised Context augmentation

In [ ]:
config = base_config.copy()
config["real_df"] = None
config["synth_df"] = syn_unsupContext_df # Unsupervised Context Augmentation
config["synth_ratio"] = 1.0 # 100% of the data is synthetic
config["generation_method"] = "unsupContext"
config["save_dataset"] = False

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))

D. Zeroshot labels

In [ ]:
config = base_config.copy()
config["real_df"] = None
config["synth_df"] = zeroshot_df # real data with labels predicted in zero-shot setting
config["synth_ratio"] = 1.0 # 100% of the data is synthetic
config["generation_method"] = "zeroshotLabels"
config["save_dataset"] = False

train_details = main_multiclassRoBERTA(**config)
results.append(get_results(train_details))


---
# Results

In [ ]:
df_results = pd.concat(results).reset_index(drop=True)
df_results.to_csv(os.path.join(FOLDER_DIR, "RoBERTA_results_dev.csv"), index=False)
display(df_results.round(3))

In [ ]:
df_results = pd.read_csv(os.path.join(FOLDER_DIR, "RoBERTA_results_dev.csv"))

plot_micromacrof1(df=df_results, x_col="synthetic_ratio", hue_col="generation_method")

In [ ]:
# df_results = pd.read_csv(os.path.join(FOLDER_DIR, "results_RoBERTA.csv"))

plot_df = df_results.set_index('method')
plot_df["train_time_norm"] = plot_df["train_time"] / max(plot_df["train_time"])
plot_df.drop(columns=["train_time", "eval_time"], inplace=True)
colors = ['#ef476f', '#f78c6b', '#ffd166', '#06d6a0', '#118ab2', '#073b4c', '#aaaaaa']
ax = plot_df.T.plot(kind='bar', figsize=(13, 5), color=colors)

plt.title('Results on dev', fontsize=16)
plt.xlabel('Metrics')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(title='Configurations', bbox_to_anchor=(1, 1), loc='upper left')
plt.grid(True)

plt.tight_layout()
plt.show()